In [1]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions 

In [2]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [3]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset.csv


In [4]:
# Import dataset
dataset = pd.read_csv("Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),Available,Battery Capacity,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_EG,Log_Available,Log_AC Recharging Speed (km/h)
0,Austria,2020,6.41,3.803179e+05,1.381911,-1.600,26797.00000,39.325301,403.164557,97.000000,57.761446,286.200000,38311.380000,2.002830,10.196083,12.848765,0.867903,NaN,4.584967,3.696979
1,Austria,2021,13.92,4.062315e+05,2.766667,2.000,39904.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,2.702703,10.594257,12.914681,1.326190,1.098612,5.129899,3.735390
2,Austria,2022,15.70,4.493822e+05,8.546870,2.700,61393.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,2.815409,11.025067,13.015631,2.256213,1.308333,5.537334,3.918300
3,Austria,2023,19.91,4.778373e+05,7.814134,0.800,76773.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.040228,11.248621,13.077028,2.176357,0.587787,5.918894,4.267004
4,Austria,2024,17.58,4.940876e+05,2.937916,0.100,102562.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,2.922086,11.538233,13.110470,1.370652,0.095310,6.314133,4.534494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,Sweden,2021,19.09,5.417760e+06,2.163197,1.300,70537.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,3.000222,11.163907,15.505193,1.151583,0.832909,5.129899,3.735390
158,Sweden,2022,32.84,5.816415e+06,8.369291,3.500,83510.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,3.521644,11.332734,15.576195,2.237437,1.504077,5.537334,3.918300
159,Sweden,2023,38.42,6.143187e+06,8.548625,1.200,126758.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.674273,11.750043,15.630854,2.256397,0.788457,5.918894,4.267004
160,Sweden,2024,34.99,6.379843e+06,2.835817,-0.300,189977.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,3.583241,12.154664,15.668654,1.344382,-0.356675,6.314133,4.534494


,Country,Year,BEV Percentage (Total Number Of Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),Available,Battery Capacity,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_EG,Log_Available,Log_AC Recharging Speed (km/h)
0,Austria,2020,6.41,3.803179e+05,1.381911,-1.600,26797.00000,39.325301,403.164557,97.000000,57.761446,286.200000,38311.380000,2.002830,10.196083,12.848765,0.867903,NaN,4.584967,3.696979
1,Austria,2021,13.92,4.062315e+05,2.766667,2.000,39904.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,2.702703,10.594257,12.914681,1.326190,1.098612,5.129899,3.735390
2,Austria,2022,15.70,4.493822e+05,8.546870,2.700,61393.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,2.815409,11.025067,13.015631,2.256213,1.308333,5.537334,3.918300
3,Austria,2023,19.91,4.778373e+05,7.814134,0.800,76773.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.040228,11.248621,13.077028,2.176357,0.587787,5.918894,4.267004
4,Austria,2024,17.58,4.940876e+05,2.937916,0.100,102562.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,2.922086,11.538233,13.110470,1.370652,0.095310,6.314133,4.534494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,Sweden,2021,19.09,5.417760e+06,2.163197,1.300,70537.00000,40.904348,493.391304,168.000000,64.046957,308.730000,40745.010000,3.000222,11.163907,15.505193,1.151583,0.832909,5.129899,3.735390
158,Sweden,2022,32.84,5.816415e+06,8.369291,3.500,83510.00000,49.314815,594.259259,253.000000,73.787037,345.440000,43874.270000,3.521644,11.332734,15.576195,2.237437,1.504077,5.537334,3.918300
159,Sweden,2023,38.42,6.143187e+06,8.548625,1.200,126758.00000,70.307692,649.230769,371.000000,79.230769,379.510785,45999.050000,3.674273,11.750043,15.630854,2.256397,0.788457,5.918894,4.267004
160,Sweden,2024,34.99,6.379843e+06,2.835817,-0.300,189977.00000,92.176355,744.736940,551.323143,88.123464,416.941976,48874.922228,3.583241,12.154664,15.668654,1.344382,-0.356675,6.314133,4.534494


In [5]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [8]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [9]:
# Split the dataset into train and test data
pre_covid_df = dataset[dataset['Year'].between(2020, 2024)].copy()
post_covid_df = dataset[dataset['Year'] >= 2025].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [10]:
# Check which columns have NaNs and how many
print("Missing Values Per Column on Train data (2020-2024)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column Test data (2025)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column on Train data (2020-2024)
DC Recharging Speed (km/h)         0
Log_Recharging Points              1
Real Range                         0
Purchase price (EUR)               0
Log_Available                      0
Log_AC Recharging Speed (km/h)     0
Battery Capacity                   0
Log_GDP                            0
Log_CPI                            1
Log_EG                            22
dtype: int64

Missing Values Per Column Test data (2025)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64


In [11]:
# Checking which rows are empty
print("Missing Log_EG in Train data (2020-2024)")
print(pre_covid_df[pre_covid_df['Log_EG'].isna()].reset_index()[['Country', 'Year', 'Log_EG']])

Missing Log_EG in Train data (2020-2024)
           Country  Year  Log_EG
0          Austria  2020     NaN
1         Bulgaria  2020     NaN
2          Croatia  2020     NaN
3   Czech Republic  2020     NaN
4          Denmark  2020     NaN
5          Estonia  2020     NaN
6          Finland  2020     NaN
7          Finland  2024     NaN
8           Greece  2020     NaN
9          Hungary  2020     NaN
10         Hungary  2024     NaN
11         Ireland  2020     NaN
12           Italy  2020     NaN
13          Latvia  2021     NaN
14          Latvia  2024     NaN
15       Lithuania  2020     NaN
16        Portugal  2020     NaN
17         Romania  2020     NaN
18         Romania  2023     NaN
19        Slovakia  2020     NaN
20           Spain  2020     NaN
21          Sweden  2020     NaN


In [12]:
features_to_impute = ['Log_Recharging Points', 'Log_EG', 'Log_CPI']

# Compute per-country median from train
train_country_medians = {}
train_global_medians = {}

for col in features_to_impute:
    train_country_medians[col] = pre_covid_df.groupby('Country')[col].median()
    train_global_medians[col] = pre_covid_df[col].median()

def robust_impute(df, country_medians, global_medians):
    df_copy = df.copy()
    for col in features_to_impute:

        # Impute per-country median
        df_copy[col] = df_copy[col].fillna(df_copy['Country'].map(country_medians[col]))

        # Fallback to global median if still NaN
        df_copy[col] = df_copy[col].fillna(global_medians[col])
    return df_copy

# Apply to train and test
pre_covid_fixed = robust_impute(pre_covid_df, train_country_medians, train_global_medians)
post_covid_fixed = robust_impute(post_covid_df, train_country_medians, train_global_medians)

In [13]:
# Verfication
print("Missing Values Per Column on Train data (2020-2024)")
print(pre_covid_fixed[features].isna().sum())

print("\nMissing Values Per Column Test data (2025)")
print(post_covid_fixed[features].isna().sum())

Missing Values Per Column on Train data (2020-2024)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64

Missing Values Per Column Test data (2025)
DC Recharging Speed (km/h)        0
Log_Recharging Points             0
Real Range                        0
Purchase price (EUR)              0
Log_Available                     0
Log_AC Recharging Speed (km/h)    0
Battery Capacity                  0
Log_GDP                           0
Log_CPI                           0
Log_EG                            0
dtype: int64


In [14]:
X_train_scaled = scaler.transform(pre_covid_fixed[features])
X_test_scaled = scaler.transform(post_covid_fixed[features])

# Convert back to DataFrame to keep column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=features)

In [15]:
# Transform the features into 2 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


In [16]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (135, 3)
Test Shape (PCs): (27, 3)


In [17]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    "Austria", "Belgium", "Czech Republic", "Denmark", "France",
    "Germany", "Hungary", "Italy", "Netherlands", "Poland",
    "Romania", "Spain", "Sweden"
]

followers_list = [
    "Bulgaria", "Croatia", "Cyprus", "Estonia", "Finland",
    "Greece", "Ireland", "Latvia", "Lithuania", "Luxembourg",
    "Malta", "Portugal", "Slovakia", "Slovenia"
]

In [18]:
# Filter Training Data
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## AUTOGLUON MODEL

**1. Explanation:** [AutoGluon](https://aws.amazon.com/blogs/opensource/machine-learning-with-autogluon-an-open-source-automl-library/)

**2. Documentation:** [Link](https://auto.gluon.ai/stable/index.html)

**3. GitHub Repository:** [Link](https://github.com/autogluon/autogluon)

In [19]:
# Prepare data function for AutoGluon: AutoGluon needs all the PCs and the target in a single DataFrame as its designed to work on Pandas dataframe.
def prepare_ag_data(X_pca, y_target):
    df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
    df['target'] = y_target.values
    return df

train_data_ag = prepare_ag_data(X_train_pca, y_train)
test_data_ag = prepare_ag_data(X_test_pca, y_test)

In [20]:
# Train AutoGluon and fit the model
predictor = TabularPredictor(
    label='target', # 'Log_BEV Percentage (Total Number Of Registrations)'
    problem_type='regression', # Prediction of continuous numbers
    eval_metric='mean_absolute_error' # 'mean_absolute_error' because R2 is unstable for single-year tests
).fit(
    train_data_ag, # Training data
    presets='best_quality', # Enables stacking and ensembling for higher accuracy
    time_limit=600          # 10-minute training limit. Will try to train as many model combinations as possible within this window.
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260318_184643"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       10.98 GB / 12.67 GB (86.7%)
Disk Space Avail:   74.86 GB / 107.72 GB (69.5%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable sta

(_ray_fit pid=3444) [1000]	valid_set's l1: 0.530227
(_ray_fit pid=3444) [2000]	valid_set's l1: 0.516057
(_ray_fit pid=3444) [3000]	valid_set's l1: 0.510887
(_ray_fit pid=3444) [4000]	valid_set's l1: 0.509313
(_ray_fit pid=3547) [1000]	valid_set's l1: 0.39012


(_dystack pid=2986) 	-0.5372	 = Validation score   (-mean_absolute_error)
(_dystack pid=2986) 	32.57s	 = Training   runtime
(_dystack pid=2986) 	0.03s	 = Validation runtime
(_dystack pid=2986) Fitting model: LightGBM_BAG_L1 ... Training model for up to 45.48s of the 88.63s of remaining time.
(_dystack pid=2986) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.16%)
(_dystack pid=2986) 	-0.562	 = Validation score   (-mean_absolute_error)
(_dystack pid=2986) 	28.31s	 = Training   runtime
(_dystack pid=2986) 	0.01s	 = Validation runtime
(_dystack pid=2986) Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 12.22s of the 55.37s of remaining time.
(_dystack pid=2986) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2986) 	-0.6009	 = Validation score   (-mean_absolute_error)
(_dystack pid=2986) 	1.53s	 = Training   runtime
(_dystack pid=2986) 	0.14s	 = Va

(_ray_fit pid=4481) [1000]	valid_set's l1: 0.466548
(_ray_fit pid=4481) [2000]	valid_set's l1: 0.441954


(_dystack pid=2986) 	-0.5182	 = Validation score   (-mean_absolute_error)
(_dystack pid=2986) 	29.37s	 = Training   runtime
(_dystack pid=2986) 	0.05s	 = Validation runtime
(_dystack pid=2986) Fitting model: LightGBM_BAG_L2 ... Training model for up to 8.82s of the 8.80s of remaining time.
(_dystack pid=2986) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.17%)
(_dystack pid=2986) 	-0.5434	 = Validation score   (-mean_absolute_error)
(_dystack pid=2986) 	29.02s	 = Training   runtime
(_dystack pid=2986) 	0.04s	 = Validation runtime
(_dystack pid=2986) Fitting model: WeightedEnsemble_L3 ... Training model for up to 129.46s of the -25.33s of remaining time.
(_dystack pid=2986) 	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/9.2 GB
(_dystack pid=2986) 	Ensemble Weights: {'LightGBMXT_BAG_L2': 1.0}
(_dystack pid=2986) 	-0.5182	 = Validation score   (-mean_absolute_error)
(_dystack pid=2986) 	0

In [21]:
# View the Model Leaderboard
# This shows you which models (XGBoost, Neural Nets, etc.) performed best within the time frame
lb = predictor.leaderboard(test_data_ag, silent=True)
print("\n--- Model Leaderboard ---")
print(lb)


--- Model Leaderboard ---
                        model  score_test  score_val          eval_metric  \
0   NeuralNetTorch_r79_BAG_L1   -0.599807  -0.538332  mean_absolute_error   
1           LightGBMXT_BAG_L1   -0.605981  -0.526384  mean_absolute_error   
2             LightGBM_BAG_L1   -0.608627  -0.550472  mean_absolute_error   
3         WeightedEnsemble_L2   -0.616942  -0.516241  mean_absolute_error   
4      RandomForestMSE_BAG_L1   -0.632124  -0.622625  mean_absolute_error   
5       NeuralNetTorch_BAG_L1   -0.656171  -0.560999  mean_absolute_error   
6        CatBoost_r177_BAG_L1   -0.657903  -0.575160  mean_absolute_error   
7        ExtraTreesMSE_BAG_L1   -0.661953  -0.623129  mean_absolute_error   
8             CatBoost_BAG_L1   -0.669877  -0.568553  mean_absolute_error   
9      NeuralNetFastAI_BAG_L1   -0.677070  -0.541974  mean_absolute_error   
10       LightGBMLarge_BAG_L1   -0.680279  -0.581111  mean_absolute_error   
11             XGBoost_BAG_L1   -0.769280  -0.574

**Explantion of the results**

The autogluon has checked on 13 different models

In [22]:
# Generate Predictions for 2020-2024
# We drop the 'target' column from the test data to predict it
# # Ensures the model is forced to guess the result using only the input features (PC1, PC2) without "cheating" by looking at the answer.
preds = predictor.predict(test_data_ag.drop(columns=['target']))

In [23]:
# Back-transform by converting log values back to real BEV percentages
y_test_real = np.exp(test_data_ag['target'])
preds_real = np.exp(preds)

In [24]:
# Create unified results dataframe
all_results = pd.DataFrame({
    'Country': post_covid_fixed['Country'],
    'Year': post_covid_fixed['Year'],
    'Actual_BEV_%': y_test_real,
    'Predicted_BEV_%': preds_real
})

# Absolute error
all_results['Error_Abs'] = (all_results['Actual_BEV_%'] - all_results['Predicted_BEV_%']).abs()

# Loop year-wise
for year, year_df in all_results.groupby('Year'):

    print(f"====>YEAR: {int(year)} PREDICTION RESULTS")

    # Leaders
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    leaders_df = (
        year_df[year_df['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(leaders_df[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    # Followers
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    followers_df = (
        year_df[year_df['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(followers_df[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


====>YEAR: 2025 PREDICTION RESULTS

**** LEADERS CLUSTER (2025) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
       Denmark         69.53        15.670119  53.859881
   Netherlands         37.68        15.413776  22.266224
        Sweden         37.58        15.884517  21.695483
       Belgium         35.40        13.532464  21.867536
       Austria         22.34        12.642888   9.697112
        France         21.01        15.875614   5.134386
       Germany         20.08        16.163254   3.916746
         Spain         10.19        12.509192   2.319192
       Hungary          9.36        15.486421   6.126421
        Poland          7.73        10.282878   2.552878
         Italy          7.18        12.419387   5.239387
       Romania          6.59        12.868287   6.278287
Czech Republic          6.55        12.643937   6.093937

**** FOLLOWERS CLUSTER (2025) ****
   Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Finland         37.81        11.763427  2

In [25]:
# Filter the results as per the clusters
all_results['Cluster'] = all_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_results_df = (
    all_results[all_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_results_df = (
    all_results[all_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_results_df.to_excel("Trial3_AutoGluon_Leaders_Results.xlsx", index=False)
followers_results_df.to_excel("Trial3_AutoGluon_Followers_Results.xlsx", index=False)